In [1]:
## This notebook shows how to construct an explained variance plot using a PMD decomposition
import masknmf
import torch
import os
import numpy as np
import sys
import matplotlib.pyplot as plt


##In the existing folder
import iblwfci_vis
import iblwfci_utils
%load_ext autoreload

Unable to find extension: VK_EXT_acquire_drm_display
Unable to find extension: VK_EXT_physical_device_drm
No config found!
EGL says it can present to the window but not natively
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),NVIDIA TITAN RTX,DiscreteGPU,Vulkan,555.42.02
❗ limited,"llvmpipe (LLVM 12.0.0, 256 bits)",CPU,Vulkan,Mesa 21.2.6 (LLVM 12.0.0)
❌,NVIDIA TITAN RTX/PCIe/SSE2,Unknown,OpenGL,3.3.0 NVIDIA 555.42.02


Max vertex attribute stride unknown. Assuming it is 2048
Max vertex attribute stride unknown. Assuming it is 2048


# Define some basic utility functions for working with factorizations (Joao's factorizations or the masknmf ideas)

In [2]:
def absorb_pixel_normalizer(U_sparse, pixel_normalizer):
    """
    U_sparse: (N_pixels, R) sparse COO
    pixel_normalizer: (N_pixels,) dense tensor

    Returns:
        U_scaled: sparse COO
    """
    U_sparse = U_sparse.coalesce()
    indices = U_sparse.indices()
    values = U_sparse.values()

    row_indices = indices[0]
    scaled_values = values * pixel_normalizer[row_indices]

    return torch.sparse_coo_tensor(
        indices,
        scaled_values,
        U_sparse.shape,
        device=U_sparse.device
    )

def absorb_mean(U_scaled_sparse, V_dense, pixelwise_mean):
    """
    U_scaled_sparse: (N, R) sparse
    V_dense: (R, T) dense
    pixelwise_mean: (N,) dense

    Returns:
        U_new (sparse), V_new (dense)
    """

    N, R = U_scaled_sparse.shape
    T = V_dense.shape[1]

    # --- append mean as new sparse column ---
    U_scaled_sparse = U_scaled_sparse.coalesce()
    indices = U_scaled_sparse.indices()
    values = U_scaled_sparse.values()

    # New column index = R
    mean_col_indices = torch.stack([
        torch.arange(N, device=pixelwise_mean.device),
        torch.full((N,), R, device=pixelwise_mean.device)
    ])

    mean_values = pixelwise_mean

    new_indices = torch.cat([indices, mean_col_indices], dim=1)
    new_values = torch.cat([values, mean_values])

    U_new = torch.sparse_coo_tensor(
        new_indices,
        new_values,
        (N, R + 1),
        device=U_scaled_sparse.device
    )

    # --- append ones row to V ---
    ones_row = torch.ones(1, T, device=V_dense.device)
    V_new = torch.cat([V_dense, ones_row], dim=0)

    return U_new.coalesce(), V_new

def compute_mean(u, v):
    return torch.sparse.mm(u, torch.mean(v, dim = 1, keepdims = True)).flatten()

In [3]:
def get_pca_spectrum_pmd_obj(pmd_obj, device = 'cpu'):
    pixelwise_normalizer = pmd_obj.var_img.to(device).flatten()
    u = pmd_obj.u.to(device)
    v = pmd_obj.v.to(device)

    print('1')
    u_absorbed  = absorb_pixel_normalizer(u, pixelwise_normalizer)
    print('2')
    u_absorbed = u_absorbed.to(device)
    print('3')
    new_mean = compute_mean(u_absorbed, v)
    print('4')
    u_final, v_final = absorb_mean(u_absorbed, v, new_mean * -1)

    u_final = u_final.to(device)
    v_final = v_final.to(device)

    print('factorization started')
    r, s, v = masknmf.compression.decomposition.compute_lowrank_factorized_svd(u_final, v_final)
    print('completed')

    s = s.cpu().numpy()
        
    #Returns the explained variance spectrum
    return np.cumsum(s**2) / np.sum(s**2)
    

In [4]:
hemocorr_pmd = masknmf.PMDArray.from_hdf5("felicia_jan_26_008/hemocorr.hdf5")
hemocorr_pmd.to('cuda')
hemocorr_pmd.rescale = True

In [5]:
parent_path = "/data/lab/IBL_Alyx/churchlandlab_ucla/Subjects/FD_24/2023-06-08/001/raw_widefield_data"
u_path = os.path.join(parent_path, "U.npy")
svt_path = os.path.join(parent_path, "SVT.npy")
svtcorr_path = os.path.join(parent_path, "SVTcorr.npy")
frames_avg_path = os.path.join(parent_path, "frames_average.npy")

manual_mask = np.load(os.path.join(parent_path, "manual_mask.npy"))

In [6]:
joao_gcamp, joao_blood, joao_hemocorr = iblwfci_utils.load_joao_results(u_path,
                  svt_path,
                  svtcorr_path,
                  frames_avg_path,
                  functional_channel = 0)

In [14]:
explained_var_spectrum_amol = get_pca_spectrum_pmd_obj(hemocorr_pmd, device = 'cuda')

In [13]:
explained_var_spectrum_joao = get_pca_spectrum_pmd_obj(joao_hemocorr, device = 'cpu') ##Use cpu here, for some reason the gpu is acting weird here

In [12]:
import numpy as np
import matplotlib.pyplot as plt

threshold = 0.95

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# ------------------ Plot 1 ------------------
explained_var = explained_var_spectrum_amol
cutoff_idx = np.argmax(explained_var >= threshold)

axes[0].plot(explained_var[:30], marker="o", label="Cumulative Explained Variance")
axes[0].axhline(y=threshold, color="red",linestyle="--", label="95% Threshold")
axes[0].axvline(x=cutoff_idx, color="gray",linestyle=":", label=f"Cutoff @ {cutoff_idx+1}")

axes[0].set_title("Hemocorrected PMD PCA spectrum", fontsize=14, weight="bold")
axes[0].set_xlabel("Number of Components", fontsize=12)
axes[0].set_ylabel("Cumulative Explained Variance", fontsize=12)
axes[0].set_xticks(range(0, 31, 2))
axes[0].set_ylim(0, 1.05)
axes[0].grid(alpha=0.3)
axes[0].legend()


# ------------------ Plot 2 ------------------
explained_var = explained_var_spectrum_joao
cutoff_idx = np.argmax(explained_var >= threshold)

axes[1].plot(explained_var[:30], marker="o", label="Cumulative Explained Variance")
axes[1].axhline(y=threshold, color="red", linestyle="--", label="95% Threshold")
axes[1].axvline(x=cutoff_idx, color="gray", linestyle=":", label=f"Cutoff @ {cutoff_idx+1}")

axes[1].set_title("Hemocorrected WFIELD PCA spectrum", fontsize=14, weight="bold")
axes[1].set_xlabel("Number of Components", fontsize=12)
axes[1].set_xticks(range(0, 31, 2))
axes[1].set_ylim(0, 1.05)
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig("Dimensionality_Comparison.png", bbox_inches="tight")
plt.show()